<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #00137cff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

#  **DS105W Mini-Project 1: London Air Quality Analysis (Winter Term 2025/2026)**

## **_Data Collection Notebook_**
- 👤 Name: Laurie Taylor
- 📛 Candidate Number: 73691
- 📅 Date: 19th February 2026
- 🎯 Purpose: Investigate Weekday vs Weekend Air Pollution Patterns Using OpenWeather API **_- Collect and Save Data._**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #c6a8ffff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

### **Introduction:**

#### Research Question: Does London's Air Clean Up on Weekends?

- What I Want to Find Out: 
    - Do London's pollutant levels drop by a larger value on weekends compared to weekdays?
    - Which pollutants have the largest difference?
    - Is there a big difference between north and south London?
    - I assume that commuter traffic helps drive the weekday-to-weekend difference in air quality; are there other pollution patterns that condradict this?

## Reproduction Instructions:

[Replace this section with your content. What steps should someone take to reproduce your analysis? What packages need to be installed? What API keys are required?]

#### Hypotheses: 

All Locations:
- **H₀:** Mean daily air pollutant levels do not differ significantly between weekdays and weekends.
- **H₁:** Mean daily air pollutant levels are significantly lower on weekends than weekdays.

Per Location:

- **Marylebone Road** (Urban Traffic): The largest weekday-weekend mean air pollutant difference, due to busy road traffic and street canyon geometry.

- **Richmond** (Suburban Background): Little-to-no weekend-weekend mean air pollutant difference, due to the location's low road density and reduced commuter traffic. PM2.5 may show a weekend increase driven by residential wood-burning stoves.

**Decisions Made in Introduction:**
- Temporal Aggregation: I will calculate the daily mean pollutant levels per pollutant from hourly data. Per-day aggregation (Monday-Sunday) is used in combination with a Mon-Fri/Sat-Sun split, to preserve daily variation. This will allow me to identify where the weekend effect begins and whether certain days have larger discrepancies than others. The weekday/weekend split will be applied in Data Analysis, once the patterns have been visualised.



#### ⚙️ **Importing additional libraries:**

</div>

In [ ]:
import os
import json
import
import time

from dotenv import load_dotenv
from datetime import datetime, timezone

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #00137cff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

## **Section 1: Testing API Authentication**

To begin this assignment, I collected data from a website called [OpenWeather](https://openweathermap.org/api). They have a wide range of APIs. The one most useful to this project is the [Air Pollution API](https://openweathermap.org/api/air-pollution). Please see below for a step-by-step process of how I tested whether the API authentication was working, followed by a personal reflection.

In [ ]:
load_dotenv()

# As long as I don't `print(api_key)`, no one will ever be able to see my key
api_key = os.getenv("API_KEY")
print(api_key[:5])

# Test that the API key works
# Collect air pollution data for right now
base_url = "http://api.openweathermap.org/data/2.5/air_pollution"

params = {



    


    "lat": 51.5074,
    "lon": -0.1278,
    "appid": api_key
}

response = requests.get(base_url, params=params)

print(f"✅ API request successful! Status code: {response.status_code}")

# What the JSON looks like:
response.json()

630f2
✅ API request successful! Status code: 200


{'coord': {'lon': -0.1257, 'lat': 51.5085},
 'list': [{'main': {'aqi': 2},
   'components': {'co': 136.95,
    'no': 0.45,
    'no2': 5.43,
    'o3': 81.16,
    'so2': 2.44,
    'pm2_5': 2.32,
    'pm10': 5.25,
    'nh3': 0.36},
   'dt': 1772110743}]}

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #c6a8ffff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

### 💭 Personal Reflection:

**What I've learned from Section 1: API Authentication & Setup**

- I've learned that API keys should be stored in an **.env** file and to load it with **dotenv**. This technqiue lets me keep my API key **private**.

- The **status code** lets me know whether the API request worked. At first, I got **401**, which meant it failed. I tried **print(api_key[:5])** to check whether it was correctly reading my key, and this came back correct. After refreshing each page and waiting a while, I got **200** which showed that it was working.

- **Package installation** can take a few tries. Even when pip show confirmed that dotenv was installed, the notebook kernel took a few attempts before it recognised the new module.

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #00137cff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

## **Section 2: Collect Historical Data**

### 2.1: Setup

In [23]:
api_key = os.getenv("API_KEY")
base_url = "http://api.openweathermap.org/data/2.5/air_pollution/history"

locations = {
    "marylebone_road": {"lat": 51.522530, "lon": -0.154611},  
    "richmond": {"lat": 51.476168, "lon": -0.230427}   
}

start = int(datetime(2020, 11, 27, tzinfo=timezone.utc).timestamp())
end = int(datetime.now(timezone.utc).timestamp())


<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #c6a8ffff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

#### Decisions Made in Setup:
**API Endpoint:**
- After viewing the [Air Pollution APIs](https://openweathermap.org/api/air-pollution), I selected the relevant page that held the historical data and used the this as my base_url, making sure to select the right endpoint: 
    - http://api.openweathermap.org/data/2.5/air_pollution/history


**Locations:**
- I chose Marylebone Road (NW1; high pollution) and Richmond (SW13; low pollution) to hopefully detect some contrast between sites with varying traffic densities. The detailed reasons are as follows:

- **Marylebone Road:** Major road that leads in and out of central London, carries roughly 90,000 vehicles per day. The tall buildings on either side of the road trap vehicle emissions that contribute to the high pollution level (street canyon geometry). Moreover, the Derpartment for Environment, Food and Rural Affairs (DEFRA) classifies Marylebone Road as an **Urban Traffic** site within the Automatic Urban and Rural Network.³ The AURN classifications help create a solid methodological grounding for comparing pollution levels.⁴

    - Rejecting Other Options: In 2020, the Strand (88 µg/m³) ranked higher than Marylebone Road (85 µg/m³) for NO₂ pollution (IQAIR).⁵ However, the Strand has undergone a pedrestrianisation scheme, and research confirms that there was a statistically significant drop in NO₂ levels by summer 2023.⁶ This would have affected the structural integrity of my dataset. It is worth noting that the Strand is also more prone to tourists than Marylebone Road (Covent Garden, Somerset House) which would likely obscure the weekday/weekend pollution composition away from regular commuter patterns. However, this is expected for a central London postcode. 

- **Richmond:** Richmond is classified as a **Suburban Background** monitoring site by DEFRA, therefore it serves as a useful low-traffic baseline for this project (low industrial activity, no airport proximity, lower road density).⁷ These conditions will help capture what pollution in London looks like when traffic is minimal.

    - Limitations: Richmond Park and the Thames corridor affect local wind patterns and therefore pollution dispersion. Furthermore, although discouraged, houses in Richmond are more likely to have wood-burning stoves than in Marylebone Road and Bloomsbury, which may inflate PM2.5 independently of traffic patterns.⁸ ⁹ I will watch for this when interpreting results.

    -  Rejecting Other Options: Croydon could have been a valid alternative to Richmond, however it sits too close to the Gatwick flight path. Air traffic ism't as likely to follow a weekday/weekend pattern aligned with commuter activity/weekly routines. This would potentially add a confounding pollution source.

**Time Horizon:**
- At this stage, I decided to collect data for all available dates, from the start (27/11/2020) up to the most recent entry (using datetime.now for easier automation). This ensured full statistical reliability for the initial datasets and less sensitivity to outliers. 

- I considered blocking the COVID-19 lockdown period out of the time horizon, because it could blur weekday/weekend pollution discrepancy levels. However, I decided to retain it in the raw datasets and defer any filtering/categorisation to the Data Transformation stage. I'll be able to make a more grounded claim as to whether this period and/or others may skew my results once the data is structured, and patterns are easier to detect. 

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #00137cff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

### **Step 3: Collect & Save Data as a JSON file in data/ Folder**

In [24]:
for name, coords in locations.items():
    print(f"Fetching {name}...")
    params = {
        "lat":   coords["lat"],
        "lon":   coords["lon"],
        "start": start,
        "end":   end,
        "appid": api_key
    }
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()

    filepath = f"../data/air_pollution_{name}.json"    
    with open(filepath, "w") as f:                
        json.dump(data, f)

    print(f"Saved {len(data['list'])} hourly records to {filepath}")
    time.sleep(1)

Fetching marylebone_road...
Saved 45326 hourly records to ../data/air_pollution_marylebone_road.json
Fetching richmond...
Saved 45326 hourly records to ../data/air_pollution_richmond.json


<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #c6a8ffff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

#### Decisions Made In Step 3:

**Pollutants:**
- At this stage, I left the API to return all available pollutants by default. I decided to keep them to avoid making false assumptions too early. I'll filter my chosen pollutants in Data Transformation and reassess my choices in Data Analysis. 

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #ffffffff; border-left: 8px solid #dfd4f2ff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:800px;color:#212121;">

**Sources:**

³ DEFRA. Automatic Urban and Rural Network (AURN). UK Air Quality Information Resource. Available at: https://uk-air.defra.gov.uk/networks/network-info?view=aurn

⁴ DEFRA. Site Information: London Marylebone Road (UKA00315). UK Air Quality Information Resource. Available at: https://uk-air.defra.gov.uk/networks/site-info?uka_id=UKA00315&search=View+Site+Information&action=site&provider=archive

⁵ IQAir (2026). Air Quality in London, United Kingdom. IQAir. Available at: https://www.iqair.com/gb/air-quality-map/uk/england/London.

⁶ Van Soesbergen, A. and Mulligan, M. (2024). 'Net impact of London Strand-Aldwych pedestrianisation project on air quality and noise.' Urban Climate, 58, 102231. DOI: 10.1016/j.uclim.2024.102231. Available at: https://www.sciencedirect.com/science/article/pii/S2212095524004280

⁷ DEFRA. Site Information: Richmond Upon Thames - Barnes Wetlands (R12). UK Air Quality Information Resource. Available at: https://uk-air.defra.gov.uk/networks/site-info?uka_id=RI2&provider=london

⁸ Font, A., Ciupek, K., Butterfield, D. and Fuller, G.W. (2022). 'Long-term trends in particulate matter from wood burning in the United Kingdom: Dependence on weather and social factors.' Environmental Pollution, 314, 120105. DOI: 10.1016/j.envpol.2022.120105. Available at: https://www.sciencedirect.com/science/article/pii/S0269749122013197

⁹ London Borough of Richmond upon Thames. Smoke Control, Domestic Wood Burners and Bonfires. Available at: https://www.richmond.gov.uk/services/environment/pollution/air_pollution/smoke_control_and_bonfires
